In [8]:
import pandas as pd
import pathlib as path
import gensim
import re
import torch

clean_train = pd.read_csv(r'..\data\cleaned_splits\processed_train.csv')
clean_val = pd.read_csv(r'..\data\cleaned_splits\processed_val.csv')
test = pd.read_csv(r"..\data\raw\testing_data.csv")

In [9]:
model_path = path.Path("../src/models/AraVec/tweets_sg_100")
aravec_model = gensim.models.Word2Vec.load(str(model_path))


In [10]:
# 2. Tokenization function
def tokenize(text: str) -> list[str]:
    """Tokenize normalized text by splitting on whitespace."""
    return str(text).strip().split()

In [11]:
from collections import Counter
# 3. Build vocabulary on TRAIN only (frequency >= 2)
word_counts = Counter()
for text in clean_train["text"]:
    word_counts.update(tokenize(text))

In [12]:
# Keep only words with count >= 2
MIN_FREQ = 2
vocab_words = [word for word, count in word_counts.items() if count >= MIN_FREQ]

In [13]:

# Special tokens: <pad> at 0, <unk> at 1
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

In [14]:
# Mapping: word -> id (stoi) and id -> word (itos)
vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for idx, word in enumerate(sorted(vocab_words), start=2):
    vocab[word] = idx
print(f"Total distinct tokens in train: {len(word_counts)}")
print(
    f"Vocabulary size (frequency >= {MIN_FREQ}) + <pad>/<unk>: {len(vocab)} types"
)

Total distinct tokens in train: 53380
Vocabulary size (frequency >= 2) + <pad>/<unk>: 16763 types


In [16]:
from IPython.display import display

# 4. Compute OOV (Out-Of-Vocabulary) rates on train, val, and test
def compute_oov(texts, vocabulary: dict) -> dict:
    total_tokens = 0
    oov_tokens = 0
    for text in texts:
        tokens = tokenize(text)
        total_tokens += len(tokens)
        oov_tokens += sum(1 for tok in tokens if tok not in vocabulary)
    oov_rate = (oov_tokens / total_tokens) if total_tokens > 0 else 0.0
    return {
        "total_tokens": total_tokens,
        "oov_tokens": oov_tokens,
        "oov_rate (%)": round(oov_rate * 100, 2),
    }
splits = {
    "train": clean_train["text"],
    "val": clean_val["text"],
    "test": test["tweet"],  # descriptive statistics only
}
oov_rows = []
for name, texts in splits.items():
    stats = compute_oov(texts, vocab)
    oov_rows.append({"split": name, **stats})
oov_table = pd.DataFrame(oov_rows)
display(oov_table)

,split,total_tokens,oov_tokens,oov_rate (%)
0,train,200476,36619,18.27
1,val,22258,5690,25.56
2,test,50556,22514,44.53


In [17]:
def text_to_ids(text: str, vocabulary: dict, max_len: int = 64) -> list[int]:
    """Convert text to a list of token IDs, truncated or padded."""
    tokens = tokenize(text)[:max_len]
    ids = [vocabulary.get(token, UNK_ID) for token in tokens]
    if not ids:
        ids = [UNK_ID]
    return ids


In [ ]:
import re
import numpy as np
import torch


# AraVec's official clean_str mapping
_TASHKEEL = re.compile(r"[\u0617-\u061A\u064B-\u0652]")
_LONGATION = re.compile(r"(.)\1+")
_SEARCH = [
    "أ",
    "إ",
    "آ",
    "ة",
    "_",
    "-",
    "/",
    ".",
    "،",
    " و ",
    " يا ",
    '"',
    "ـ",
    "'",
    "ى",
    "\\",
    "\n",
    "\t",
    "&quot;",
    "?",
    "؟",
    "!",
]
_REPLACE = [
    "ا",
    "ا",
    "ا",
    "ه",
    " ",
    " ",
    "",
    "",
    "",
    " و",
    " يا",
    "",
    "",
    "",
    "ي",
    "",
    " ",
    " ",
    " ",
    " ? ",
    " ؟ ",
    " ! ",
]


def aravec_clean_str(text: str) -> str:
    cleaned = _TASHKEEL.sub("", str(text))
    cleaned = _LONGATION.sub(r"\1\1", cleaned)
    cleaned = cleaned.replace("وو", "و").replace("يي", "ي").replace("اا", "ا")
    for s, r in zip(_SEARCH, _REPLACE):
        cleaned = cleaned.replace(s, r)
    return cleaned.strip()


EMBED_DIM = 100
rng = np.random.default_rng(42)

# 1. Random embedding matrix
random_matrix = rng.normal(scale=0.02, size=(len(vocab), EMBED_DIM)).astype(
    np.float32
)
random_matrix[PAD_ID] = 0.0

# 2. AraVec embedding matrix
aravec_matrix = rng.normal(scale=0.02, size=(len(vocab), EMBED_DIM)).astype(
    np.float32
)
aravec_matrix[PAD_ID] = 0.0

hits = 0
for word, idx in vocab.items():
    if idx in (PAD_ID, UNK_ID):
        continue
    cleaned_word = aravec_clean_str(word)
    if cleaned_word in aravec_model.wv:
        aravec_matrix[idx] = aravec_model.wv[cleaned_word]
        hits += 1

print(
    f"AraVec coverage on vocabulary: {hits}/{len(vocab)} ({hits / len(vocab) * 100:.1f}%)"
)


AraVec coverage on vocabulary: 15182/16763 (90.6%)


In [19]:
from torch.utils.data import DataLoader, Dataset

# Set device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


def encode_text(text: str, max_len: int = 64) -> list[int]:
    tokens = tokenize(text)[:max_len]
    ids = [vocab.get(tok, UNK_ID) for tok in tokens]
    return ids if ids else [UNK_ID]


class SarcasmDataset(Dataset):

    def __init__(self, df: pd.DataFrame, max_len: int = 64):
        self.rows = [encode_text(t, max_len) for t in df["text"]]
        self.labels = df["sarcasm"].astype(bool).astype(int).values

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx], self.labels[idx]


def pad_collate_fn(batch):
    sequences, labels = zip(*batch)
    batch_max_len = max(len(seq) for seq in sequences)

    padded = torch.zeros(len(sequences), batch_max_len, dtype=torch.long)
    mask = torch.zeros(len(sequences), batch_max_len, dtype=torch.bool)

    for i, seq in enumerate(sequences):
        padded[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
        mask[i, : len(seq)] = True  # True for actual words, False for padding

    return padded, mask, torch.tensor(labels, dtype=torch.long)


BATCH_SIZE = 64
train_loader = DataLoader(
    SarcasmDataset(clean_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=pad_collate_fn,
)
val_loader = DataLoader(
    SarcasmDataset(clean_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=pad_collate_fn,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


Using device: cpu
Train batches: 175 | Val batches: 20


In [20]:
import copy
import math
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# --- 1. Sinusoidal Positional Encoding for Transformer ---
class SinusoidalPositionalEncoding(nn.Module):

    def __init__(self, d_model: int = 128, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


# --- 2. C1: BiLSTM with Max-Pooling ---
class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 100,
        hidden_dim: int = 128,
        dropout: float = 0.3,
        pretrained_weights=None,
        freeze: bool = False,
    ):
        super().__init__()
        if pretrained_weights is not None:
            weights = torch.tensor(pretrained_weights, dtype=torch.float32)
            self.embedding = nn.Embedding.from_pretrained(
                weights, freeze=freeze, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, batch_first=True, bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self.dropout(self.embedding(ids))
        states, _ = self.lstm(x)  # shape: (batch, seq_len, 2 * hidden_dim)

        # Max-pooling across time, masking out pad tokens
        filled = states.masked_fill(~mask.unsqueeze(-1), -1e9)
        pooled = filled.max(dim=1).values
        return self.fc(self.dropout(pooled))


# --- 3. C2: BiLSTM + Additive Attention Pooling ---
class BiLSTMAttention(nn.Module):
    """Additive attention: s_i = v^T tanh(W h_i), alpha = softmax(s), c = sum(alpha_i * h_i)"""

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 100,
        hidden_dim: int = 128,
        dropout: float = 0.3,
        pretrained_weights=None,
        freeze: bool = False,
    ):
        super().__init__()
        if pretrained_weights is not None:
            weights = torch.tensor(pretrained_weights, dtype=torch.float32)
            self.embedding = nn.Embedding.from_pretrained(
                weights, freeze=freeze, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, batch_first=True, bidirectional=True
        )
        # Attention projection W and context vector v
        self.W = nn.Linear(hidden_dim * 2, hidden_dim * 2, bias=False)
        self.v = nn.Linear(hidden_dim * 2, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(
        self,
        ids: torch.Tensor,
        mask: torch.Tensor,
        return_attention: bool = False,
    ):
        x = self.dropout(self.embedding(ids))
        states, _ = self.lstm(x)  # (batch, seq_len, 256)

        # Compute additive attention weights
        scores = self.v(torch.tanh(self.W(states))).squeeze(-1)  # (batch, seq_len)
        scores = scores.masked_fill(~mask, -1e9)
        alpha = torch.softmax(scores, dim=-1)  # (batch, seq_len)

        # Weighted context vector c
        context = torch.bmm(alpha.unsqueeze(1), states).squeeze(1)  # (batch, 256)
        logits = self.fc(self.dropout(context))

        if return_attention:
            return logits, alpha
        return logits


# --- 4. C3 & C3 Ablation: Mini-Transformer Encoder ---
class MiniTransformer(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 100,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 2,
        dropout: float = 0.3,
        use_pe: bool = True,  # False for C3 ablation
        pretrained_weights=None,
        freeze: bool = False,
    ):
        super().__init__()
        if pretrained_weights is not None:
            weights = torch.tensor(pretrained_weights, dtype=torch.float32)
            self.embedding = nn.Embedding.from_pretrained(
                weights, freeze=freeze, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.proj = nn.Linear(embed_dim, d_model)
        self.use_pe = use_pe
        if use_pe:
            self.pe = SinusoidalPositionalEncoding(d_model=d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, 2)

    def forward(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self.proj(self.embedding(ids))
        if self.use_pe:
            x = self.pe(x)
        x = self.dropout(x)

        # src_key_padding_mask: True for padding positions (to be ignored)
        encoded = self.encoder(x, src_key_padding_mask=~mask)

        # Mean pooling over valid (non-padded) positions
        weights = mask.unsqueeze(-1).float()
        pooled = (encoded * weights).sum(dim=1) / weights.sum(dim=1).clamp(
            min=1.0
        )
        return self.fc(self.dropout(pooled))


Using device: cpu


In [21]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 8,
    lr: float = 1e-3,
    patience: int = 2,
    device: torch.device = device,
    criterion: nn.Module = None,
    threshold: float = 0.5,
):
    model.to(device)
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    # Trainable parameters only (skips frozen embedding weights)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr)

    best_val_f1 = -1.0
    best_state = None
    best_epoch = 0
    patience_counter = 0
    best_metrics = {}
    best_val_probas = None

    print(f"Training {model.__class__.__name__} (max {epochs} epochs)...")

    for epoch in range(1, epochs + 1):
        # 1. Training Phase
        model.train()
        total_loss = 0.0
        for ids, mask, labels in train_loader:
            ids, mask, labels = (
                ids.to(device),
                mask.to(device),
                labels.to(device),
            )

            optimizer.zero_grad()
            logits = model(ids, mask)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * len(labels)

        train_loss = total_loss / len(train_loader.dataset)

        # 2. Validation Phase
        model.eval()
        val_targets, val_probas = [], []
        with torch.no_grad():
            for ids, mask, labels in val_loader:
                ids, mask = ids.to(device), mask.to(device)
                logits = model(ids, mask)
                probs = torch.softmax(logits, dim=-1)[:, 1]
                val_probas.extend(probs.cpu().numpy())
                val_targets.extend(labels.numpy())

        val_targets = np.array(val_targets)
        val_probas = np.array(val_probas)
        val_preds = (val_probas >= threshold).astype(int)

        # Sarcastic class F1 (pos_label=1)
        val_sarcasm_f1 = f1_score(
            val_targets, val_preds, pos_label=1, zero_division=0
        )
        val_macro_f1 = f1_score(
            val_targets, val_preds, average="macro", zero_division=0
        )
        val_acc = accuracy_score(val_targets, val_preds)

        print(
            f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | "
            f"Val F1 (sarcasm): {val_sarcasm_f1:.4f} | Val Macro-F1: {val_macro_f1:.4f}"
        )

        # 3. Early Stopping Check
        if val_sarcasm_f1 > best_val_f1:
            best_val_f1 = val_sarcasm_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            best_val_probas = val_probas
            best_metrics = {
                "val_sarcasm_f1": val_sarcasm_f1,
                "val_macro_f1": val_macro_f1,
                "val_precision": precision_score(
                    val_targets, val_preds, pos_label=1, zero_division=0
                ),
                "val_recall": recall_score(
                    val_targets, val_preds, pos_label=1, zero_division=0
                ),
                "val_accuracy": val_acc,
                "best_epoch": best_epoch,
            }
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"--> Early stopping triggered at epoch {epoch}!")
                break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    print(
        f"✓ Finished. Best Val F1 (sarcasm) = {best_val_f1:.4f} at epoch {best_epoch}\n"
    )
    return best_metrics, best_val_probas


In [22]:
results = []
VOCAB_SIZE = len(vocab)

# 1. C1 with Random Embedding
m_c1_rand = BiLSTMClassifier(VOCAB_SIZE, pretrained_weights=None)
res_c1_rand, _ = train_model(m_c1_rand, train_loader, val_loader)
results.append(
    {"Model": "C1 BiLSTM", "Embedding": "Random", "Ablation": "None", **res_c1_rand}
)

# 2. C1 with AraVec Frozen
m_c1_frozen = BiLSTMClassifier(
    VOCAB_SIZE, pretrained_weights=aravec_matrix, freeze=True
)
res_c1_frozen, _ = train_model(m_c1_frozen, train_loader, val_loader)
results.append(
    {
        "Model": "C1 BiLSTM",
        "Embedding": "AraVec (Frozen)",
        "Ablation": "None",
        **res_c1_frozen,
    }
)

# 3. C1 with AraVec Unfrozen
m_c1_unfrozen = BiLSTMClassifier(
    VOCAB_SIZE, pretrained_weights=aravec_matrix, freeze=False
)
res_c1_unfrozen, _ = train_model(m_c1_unfrozen, train_loader, val_loader)
results.append(
    {
        "Model": "C1 BiLSTM",
        "Embedding": "AraVec (Unfrozen)",
        "Ablation": "None",
        **res_c1_unfrozen,
    }
)

# 4. C2 BiLSTM + Attention (with AraVec Unfrozen)
m_c2 = BiLSTMAttention(
    VOCAB_SIZE, pretrained_weights=aravec_matrix, freeze=False
)
res_c2, _ = train_model(m_c2, train_loader, val_loader)
results.append(
    {
        "Model": "C2 BiLSTM + Attention",
        "Embedding": "AraVec (Unfrozen)",
        "Ablation": "None",
        **res_c2,
    }
)

# 5. C3 Mini-Transformer with Sinusoidal PE
m_c3 = MiniTransformer(
    VOCAB_SIZE, use_pe=True, pretrained_weights=aravec_matrix, freeze=False
)
res_c3, _ = train_model(m_c3, train_loader, val_loader)
results.append(
    {
        "Model": "C3 Transformer",
        "Embedding": "AraVec (Unfrozen)",
        "Ablation": "With PE",
        **res_c3,
    }
)

# 6. C3 Ablation: Mini-Transformer WITHOUT PE
m_c3_no_pe = MiniTransformer(
    VOCAB_SIZE, use_pe=False, pretrained_weights=aravec_matrix, freeze=False
)
res_c3_no_pe, _ = train_model(m_c3_no_pe, train_loader, val_loader)
results.append(
    {
        "Model": "C3 Transformer",
        "Embedding": "AraVec (Unfrozen)",
        "Ablation": "Without PE",
        **res_c3_no_pe,
    }
)

# Display Summary Table
df_results = pd.DataFrame(results)
df_results


Training BiLSTMClassifier (max 8 epochs)...
Epoch 01 | Train Loss: 0.4318 | Val F1 (sarcasm): 0.1197 | Val Macro-F1: 0.5142
Epoch 02 | Train Loss: 0.3915 | Val F1 (sarcasm): 0.1423 | Val Macro-F1: 0.5227
Epoch 03 | Train Loss: 0.3560 | Val F1 (sarcasm): 0.2379 | Val Macro-F1: 0.5729
Epoch 04 | Train Loss: 0.3124 | Val F1 (sarcasm): 0.3743 | Val Macro-F1: 0.6374
Epoch 05 | Train Loss: 0.2760 | Val F1 (sarcasm): 0.4000 | Val Macro-F1: 0.6550
Epoch 06 | Train Loss: 0.2400 | Val F1 (sarcasm): 0.4274 | Val Macro-F1: 0.6667
Epoch 07 | Train Loss: 0.2014 | Val F1 (sarcasm): 0.4413 | Val Macro-F1: 0.6751
Epoch 08 | Train Loss: 0.1736 | Val F1 (sarcasm): 0.4371 | Val Macro-F1: 0.6750
✓ Finished. Best Val F1 (sarcasm) = 0.4413 at epoch 7

Training BiLSTMClassifier (max 8 epochs)...
Epoch 01 | Train Loss: 0.4074 | Val F1 (sarcasm): 0.1299 | Val Macro-F1: 0.5205
Epoch 02 | Train Loss: 0.3559 | Val F1 (sarcasm): 0.2804 | Val Macro-F1: 0.5963
Epoch 03 | Train Loss: 0.3437 | Val F1 (sarcasm): 0.2510 

c:\ghoufrane\Master\M2\llms\TP2\.venv\Lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


Epoch 01 | Train Loss: 0.4026 | Val F1 (sarcasm): 0.3661 | Val Macro-F1: 0.6405
Epoch 02 | Train Loss: 0.3058 | Val F1 (sarcasm): 0.4000 | Val Macro-F1: 0.6574
Epoch 03 | Train Loss: 0.2307 | Val F1 (sarcasm): 0.4583 | Val Macro-F1: 0.6870
Epoch 04 | Train Loss: 0.1479 | Val F1 (sarcasm): 0.4752 | Val Macro-F1: 0.6899
Epoch 05 | Train Loss: 0.0911 | Val F1 (sarcasm): 0.4232 | Val Macro-F1: 0.6653
Epoch 06 | Train Loss: 0.0589 | Val F1 (sarcasm): 0.4703 | Val Macro-F1: 0.6864
--> Early stopping triggered at epoch 6!
✓ Finished. Best Val F1 (sarcasm) = 0.4752 at epoch 4

Training MiniTransformer (max 8 epochs)...
Epoch 01 | Train Loss: 0.3871 | Val F1 (sarcasm): 0.3564 | Val Macro-F1: 0.6337
Epoch 02 | Train Loss: 0.3039 | Val F1 (sarcasm): 0.4611 | Val Macro-F1: 0.6870
Epoch 03 | Train Loss: 0.2283 | Val F1 (sarcasm): 0.5390 | Val Macro-F1: 0.7224
Epoch 04 | Train Loss: 0.1464 | Val F1 (sarcasm): 0.5000 | Val Macro-F1: 0.7011
Epoch 05 | Train Loss: 0.0980 | Val F1 (sarcasm): 0.4939 | Va

,Model,Embedding,Ablation,val_sarcasm_f1,val_macro_f1,val_precision,val_recall,val_accuracy,best_epoch
0,C1 BiLSTM,Random,None,0.441261,0.675133,0.562044,0.363208,0.843499,7
1,C1 BiLSTM,AraVec (Frozen),None,0.385135,0.651129,0.678571,0.268868,0.853933,6
2,C1 BiLSTM,AraVec (Unfrozen),None,0.461095,0.686958,0.592593,0.377358,0.849920,4
3,C2 BiLSTM + Attention,AraVec (Unfrozen),None,0.447205,0.682589,0.654545,0.339623,0.857143,4
4,C3 Transformer,AraVec (Unfrozen),With PE,0.475196,0.689945,0.532164,0.429245,0.838684,4
5,C3 Transformer,AraVec (Unfrozen),Without PE,0.539007,0.722379,0.540284,0.537736,0.843499,3
